In [1]:
import mdtraj as md
import numpy as np
# import logging
import os
import subprocess
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
t = 300
ints = list(range(54, 151, 2))
distances_in_nm = [f"{d/100:.2f}" for d in ints]

output_file = "/gibbs/arghavan/hp-results-pc/plate_box_dimensions.dat"

with open(output_file, "w") as out:

    out.write("Box dimensions for each plate simulation\n")
    out.write(
        f"{'distance(Å)':<12}"
        f"{'x(Å)':<12}"
        f"{'y(Å)':<12}"
        f"{'z(Å)':<12}\n"
    )

    for d in distances_in_nm:

        distance_angstrom = float(d) * 10

        top_path = (
            f"/gibbs/arghavan/plate_simulations/"
            f"{t}K/{d}/topol_edited.prmtop"
        )

        if not os.path.exists(top_path):
            print(f"Missing: {top_path}")
            continue

        # cmd = (
        #     f"grep -A 2 '%FLAG BOX_DIMENSIONS' {top_path} "
        #     f"| tail -n 1"
        # )
        cmd = (
            f"awk '/%FLAG BOX_DIMENSIONS/{{getline; getline; print; exit}}' "
            f"{top_path}"
        )

        result = subprocess.run(
            cmd,
            shell=True,
            capture_output=True,
            text=True
        )

        if result.returncode != 0 or not result.stdout.strip():
            print(f"Could not find BOX_DIMENSIONS in {top_path}")
            continue

        values = result.stdout.split()

        beta = float(values[0])  # not used
        x = float(values[1])
        y = float(values[2])
        z = float(values[3])

        out.write(
            f"{distance_angstrom:<12.1f}"
            f"{x:<12.3f}"
            f"{y:<12.3f}"
            f"{z:<12.3f}\n"
        )

print(f"Results written to {output_file}")

Results written to /gibbs/arghavan/hp-results-pc/plate_box_dimensions.dat


In [ ]:
temperatures = [f"{T}" for T in range(300, 301, 20)]
ints = list(range(54,151,2))
distances_in_nm = [f"{d/100:.2f}" for d in ints]

In [2]:
t = 300
d = '0.94'
d_in_nm = round(float(d), 2)
d_in_angstrom = round(float(d), 2) * 10
input_path = f'/gibbs/arghavan/plate_simulations/{t}K/{d}/'
traj_path = f'{input_path}{t}K_{d}.nc'
top_path = f'{input_path}topol_edited.prmtop'

traj = md.load(traj_path, top=top_path)
tpl = traj.top

In [9]:
all_ox_indices = tpl.select("name O")
plate_indices = tpl.select("resname =~ 'WALL'")
wall_coords = traj.xyz[0][plate_indices]
x_center = 2.50
x_min = np.float32(x_center - d_in_nm / 2) * 10
x_max = np.float32(x_center + d_in_nm / 2) * 10
y_min = round(np.min(wall_coords[:, 1]) * 10, 2)
y_max = round(np.max(wall_coords[:, 1]) * 10, 2)
z_min = round(np.min(wall_coords[:, 2]) * 10, 2)
z_max = round(np.max(wall_coords[:, 2]) * 10, 2)

print (x_min, x_max)
print (y_min, y_max)
print (z_min, z_max)

20.4 29.6
16.83 33.93
31.19 48.81


parm /gibbs/arghavan/plate_simulations/{t}K/{d}/topol_edited.prmtop
trajin /gibbs/arghavan/plate_simulations/{t}K/{d}/{t}K_{d}.nc

gist refdens refdens{t} temp {t} gridspacn 0.10 gridcntr 25 25.38 40 griddim {float(d)*100} 170 170 floatfmt double floatprec 8 prefix gZ info Info.dat
go

gist pme refdens 0.03331284139 temp 300 gridspacn 0.10 gridcntr 25 25.38 40 griddim 504 500 796 floatfmt double floatprec 8 prefix d-{d_in_angstrom} info Info.dat
go

parm /path_to_prmtop
trajin /path_to_traj
gist pme refdens 0.03331284139 temp 300 gridspacn 0.10 gridcntr 25 25.38 40 griddim 504 500 796 floatfmt double floatprec 8 prefix d-{d_in_angstrom} info Info.dat
go